# AgriSmart: Five-Model Predictive Analytics

This notebook loads the processed dataset created by `prepare_dataset.ipynb`, trains five regression models, and compares them using MAE, MSE, RMSE, and R². It saves the evaluation table and Linear Regression test predictions in the `results` folder.

**Run `prepare_dataset.ipynb` first.** Open Jupyter with `AgriSmart_Corrected` as the working directory.

## 1. Install required libraries

In [3]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import libraries and configure project paths

In [4]:
"""AgriSmart Predictive Analytics: train and compare five regression models.

Run this notebook from the AgriSmart_Corrected project root after running prepare_dataset.ipynb.

Input:  datasets/processed/crop_data_v2.csv
Output: results/model_evaluation_results.csv
        results/test_set_predictions.csv

This is a random holdout evaluation on a single-date APMC snapshot. It is not
an evaluation of future-date forecasting performance.
"""
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Set the project folder
PROJECT_DIR = Path.cwd()
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "datasets" / "processed" / "crop_data_v2.csv").exists():
    candidates = [p for p in PROJECT_DIR.parents if (p / "datasets" / "processed" / "crop_data_v2.csv").exists()]
    if not candidates:
        raise FileNotFoundError(
            "Could not find datasets/processed/crop_data_v2.csv. Run prepare_dataset.ipynb first."
        )
    PROJECT_DIR = candidates[0]
DATA_FILE = PROJECT_DIR / "datasets" / "processed" / "crop_data_v2.csv"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
TARGET = "Modal_Price"

categorical_features = ["Commodity", "Variety", "Grade", "State", "District", "Market"]
numerical_features = ["Area", "Production", "Yield"]
feature_columns = categorical_features + numerical_features



## 3. Train and evaluate the five regression models

The models are Linear Regression, Decision Tree, Random Forest, Gradient Boosting, and Support Vector Regression.

In [5]:
def main():

    data = pd.read_csv(DATA_FILE)
    data.columns = data.columns.str.strip()
    if TARGET not in data.columns:
        raise ValueError(f"Target column {TARGET!r} is missing.")
    missing = [c for c in feature_columns if c not in data.columns]
    if missing:
        raise ValueError(f"Required feature columns are missing: {missing}")

    data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
    data = data.dropna(subset=[TARGET])
    data = data[data[TARGET] > 0].copy()
    X, y = data[feature_columns].copy(), data[TARGET].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(
            max_depth=12, min_samples_split=5, min_samples_leaf=2, random_state=42
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=300, max_depth=15, min_samples_split=5,
            min_samples_leaf=2, random_state=42, n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=3,
            loss="squared_error", random_state=42
        ),
        "Support Vector Regression": SVR(
            kernel="rbf", C=100, epsilon=0.1, gamma="scale"
        ),
    }

    rows, prediction_frames = [], []
    for name, estimator in models.items():
        preprocessor = ColumnTransformer([
            ("categorical", categorical_pipeline, categorical_features),
            ("numerical", numerical_pipeline, numerical_features),
        ], remainder="drop")
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", estimator),
        ])
        pipeline.fit(X_train, y_train)
        predicted = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, predicted)
        rows.append({
            "Model": name,
            "MAE": mean_absolute_error(y_test, predicted),
            "MSE": mse,
            "RMSE": mse ** 0.5,
            "R2 Score": r2_score(y_test, predicted),
        })
        if name == "Linear Regression":
            prediction_frames.append(pd.DataFrame({
                "Actual_Modal_Price": y_test.to_numpy(),
                "Predicted_Modal_Price": predicted,
                "Absolute_Error": abs(y_test.to_numpy() - predicted),
            }))

    results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    results.to_csv(RESULTS_DIR / "model_evaluation_results.csv", index=False)
    prediction_frames[0].to_csv(RESULTS_DIR / "test_set_predictions.csv", index=False)

    print("\nAGRISMART: FIVE-MODEL PREDICTIVE ANALYTICS")
    print(f"Dataset: {len(data):,} rows; train: {len(X_train):,}; test: {len(X_test):,}")
    print("Target: Modal_Price (Indian rupees per quintal)")
    print("Predictors: six categorical and three numerical features")
    print("Excluded: Min_Price, Max_Price, constant Year and Season")
    print("\nModel comparison (lower MAE/RMSE is better; higher R² is better):")
    print(results.to_string(index=False, formatters={
        "MAE": lambda v: f"{v:,.2f}",
        "MSE": lambda v: f"{v:,.2f}",
        "RMSE": lambda v: f"{v:,.2f}",
        "R2 Score": lambda v: f"{v:.4f}",
    }))
    print(f"\nBest by RMSE: {results.iloc[0]['Model']}")
    print(f"Saved: {RESULTS_DIR / 'model_evaluation_results.csv'}")
    print("Important: random split on a single date does not validate future forecasting.")

main()


AGRISMART: FIVE-MODEL PREDICTIVE ANALYTICS
Dataset: 2,444 rows; train: 1,955; test: 489
Target: Modal_Price (Indian rupees per quintal)
Predictors: six categorical and three numerical features
Excluded: Min_Price, Max_Price, constant Year and Season

Model comparison (lower MAE/RMSE is better; higher R² is better):
                    Model      MAE           MSE     RMSE R2 Score
        Linear Regression 1,147.05  4,549,118.17 2,132.87   0.7550
        Gradient Boosting 1,451.33  5,751,124.89 2,398.15   0.6903
            Random Forest 1,636.86  7,023,112.09 2,650.12   0.6218
            Decision Tree 1,849.93  8,800,218.39 2,966.52   0.5261
Support Vector Regression 1,886.15 18,220,347.23 4,268.53   0.0189

Best by RMSE: Linear Regression
Saved: C:\Users\bansa\Downloads\AgriSmart_Predictive_Analytics_Jupyter_Notebooks (1)\AgriSmart_Corrected\results\model_evaluation_results.csv
Important: random split on a single date does not validate future forecasting.
